# cv_routes — Segmentation de routes IGN (FLAIR-INC)
**ArcGIS Online Notebooks** · Exécuter les cellules dans l'ordre.

## 1 · Installation & modèle

In [ ]:
import subprocess, sys, zipfile, urllib.request, pathlib

# 1. Dépendances géospatiales + onnxruntime (conda-forge)
subprocess.run(["conda","install","-c","conda-forge","-y","--quiet",
                "rasterio","geopandas","shapely","fiona","onnxruntime","huggingface_hub"],
               check=True)

# 2. Téléchargement et extraction de cv_routes (pas de pip install, pas de git)
LIB_DIR = pathlib.Path("/arcgis/home/cv-routes-lib")
if not LIB_DIR.exists():
    zip_path = pathlib.Path("/arcgis/home/cv-routes.zip")
    urllib.request.urlretrieve(
        "https://github.com/DIGIT-Seine-Ouest/cv-routes-bitumees/archive/refs/heads/main.zip",
        zip_path,
    )
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(LIB_DIR)

sys.path.insert(0, str(next(LIB_DIR.iterdir())))  # cv-routes-bitumees-main/

# 3. Téléchargement du modèle ONNX (HuggingFace Hub, mis en cache)
from cv_routes import FlairModel, download_model

MODEL_PATH = download_model(dest="/arcgis/home/flair_12cl_resnet34_unet.onnx")
model      = FlairModel(MODEL_PATH)
print(f"✓ Prêt — {model.NUM_CLASSES} classes FLAIR")

## 2 · Téléchargement des tuiles WMS

In [ ]:
from cv_routes import fetch_tiles, CITIES

# Communes disponibles : BOULOGNE-BILLANCOURT, CHAVILLE, ISSY-LES-MOULINEAUX,
#                        MARNES-LA-COQUETTE, MEUDON, SEVRES, VANVES, VILLE-D'AVRAY
COMMUNE   = "MEUDON"
TILES_DIR = f"/arcgis/home/tiles/{COMMUNE}"

def progress(r, m): print(f"\r  {'█'*int(r*30)+'░'*(30-int(r*30))}  {m}", end="", flush=True)

tiles = fetch_tiles(COMMUNE, TILES_DIR, on_progress=progress)
print(f"\n✓ {len(tiles)} tuile(s)")

## 3 · Inférence commune entière + exports

In [ ]:
from cv_routes import run_on_city

EXPORT_DIR = f"/arcgis/home/exports/{COMMUNE}"

result = run_on_city(model, COMMUNE, tiles, export_dir=EXPORT_DIR, on_progress=progress)

print(f"\n✓ {result['n_tiles']} tuiles — route moy. {result['road_pct']:.1f}%")
print(f"  GeoPackage : {result['gpkg']}")
print(f"  ZIP TIFFs  : {result['tifs_zip']}")

## 4 · Visualisation

In [ ]:
import matplotlib.pyplot as plt

n = min(4, len(result["tile_results"]))
fig, axes = plt.subplots(n, 3, figsize=(13, 4*n))
if n == 1: axes = [axes]

for i, tr in enumerate(result["tile_results"][:n]):
    axes[i][0].imshow(tr["ortho"])
    axes[i][1].imshow(tr["flair_map"])
    axes[i][2].imshow(tr["road_overlay"])
    axes[i][0].set_ylabel(f"tuile {i}", fontsize=9)
    axes[i][2].set_title(f"{tr['road_pct']:.1f}%", fontsize=9)

axes[0][0].set_title("Ortho")
axes[0][1].set_title("FLAIR 12 classes")
axes[0][2].set_title("Routes FLAIR")
for row in axes:
    for ax in row: ax.axis("off")
plt.suptitle(COMMUNE, fontsize=13)
plt.tight_layout(); plt.show()

## 5 · Lecture des exports

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(result["gpkg"], layer="route_flair")
print(f"Polygones : {len(gdf)}  |  CRS : {gdf.crs}  |  Surface : {gdf.area.sum()/1e4:.1f} ha")

fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, color="orange", alpha=0.75, edgecolor="none")
ax.set_title(f"Routes FLAIR — {COMMUNE}")
plt.tight_layout(); plt.show()